### The dataset we have does not contain the match score column in it which is important for ATS systems, <br> and for training our model as well so, we will create that column using cosine similarity.

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\HireMind\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7597.44it/s]


In [3]:
df = pd.read_csv('dataset/train.csv')
df.head(5)

,resume_text,job_description_text,label
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,No Fit
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,No Fit
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,No Fit
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",No Fit
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,No Fit


In [5]:
scores = []

for i in range(len(df)):
    resume = df.loc[i, "resume_text"]
    jd = df.loc[i, "job_description_text"]

    # Generate embeddings
    resume_embedding = model.encode([resume])
    jd_embedding = model.encode([jd])

    # Compute cosine similarity
    similarity = cosine_similarity(resume_embedding, jd_embedding)[0][0]

    # Normalize to 0-1 range
    normalized_score = (similarity + 1) / 2

    scores.append(round(normalized_score, 4))


In [6]:
# Add new column
df["match_score"] = scores

# Save dataset
df.to_csv("dataset/resumeJD_pairs.csv", index=False)

print(df.head())

                                         resume_text  \
0  SummaryHighly motivated Sales Associate with e...   
1  Professional SummaryCurrently working with Cat...   
2  SummaryI started my construction career in Jun...   
3  SummaryCertified Electrical Foremanwith thirte...   
4  SummaryWith extensive experience in business/r...   

                                job_description_text   label  match_score  
0  Net2Source Inc. is an award-winning total work...  No Fit       0.7132  
1  At Salas OBrien we tell our clients that were ...  No Fit       0.6445  
2  Schweitzer Engineering Laboratories (SEL) Infr...  No Fit       0.7522  
3  Mizick Miller & Company, Inc. is looking for a...  No Fit       0.6135  
4  Life at Capgemini\nCapgemini supports all aspe...  No Fit       0.6612  
